## 🎯 Learning Objectives
* Design a multi-agent system to automate a complex business process.
* Define distinct agent roles, responsibilities, and communication protocols.
* Incorporate architectural patterns such as sequential and conditional workflows.
* Integrate human-in-the-loop mechanisms for critical decision points.
* Implement robust state management for long-running agentic processes.
* Discuss considerations for deploying such a system in a production environment.


## Exercise: Architect a Multi-Agent Workflow for Customer Support

**Lesson ID:** AG02-L09

Welcome to this practical exercise! In this lesson, you will apply the architectural design patterns and practices we've discussed to a real-world business scenario. Your task is to design and partially implement a multi-agent system to automate a customer support ticket resolution process.

### Scenario: Streamlining Customer Support

A growing tech company receives a high volume of customer support tickets daily. Many of these tickets are routine (e.g., password resets, common FAQs), but a significant portion requires specialized knowledge or human intervention (e.g., complex technical bugs, billing disputes). The current manual process is slow and resource-intensive. The company aims to leverage agentic AI to streamline this, reducing human workload while ensuring high-quality and timely resolutions.

### Business Process Stages:

1.  **Ticket Ingestion:** A new customer support ticket arrives (e.g., via email, web form).
2.  **Triage & Categorization:** The ticket's subject and description are analyzed to understand the core issue, categorize it (e.g., "Billing", "Technical Bug", "Feature Request", "General Inquiry"), and assign an initial priority (e.g., "Low", "Medium", "High", "Critical").
3.  **Information Gathering & Initial Solution Search:** Based on the category, relevant information is gathered. This might involve searching an internal knowledge base, querying customer CRM data, or checking recent system outages.
4.  **Solution Formulation:** A draft resolution or response is generated based on all available information.
5.  **Human Review/Escalation:** For complex, high-priority, or sensitive issues, the drafted solution or the entire ticket is flagged for human review or direct escalation to a human agent. Otherwise, it proceeds to auto-resolution.
6.  **Resolution/Response:** The final resolution is sent to the customer.

### Your Task:

#### Part 1: Design (Conceptual)

1.  **Agent Identification:** Identify at least **3 distinct agent roles** required for this workflow. For each agent:
    *   Describe its primary responsibility.
    *   List its capabilities (e.g., tools it might use like "Knowledge Base Search", "CRM Lookup", "LLM for summarization").
    *   Specify its expected input and output.
2.  **Workflow Orchestration:** Illustrate the workflow using a high-level textual description or pseudocode. Clearly show the sequence of agents, conditional branching (e.g., based on priority), and communication patterns between them.
3.  **State Management:** Explain how the state of a customer ticket will be managed and passed between agents throughout the workflow. What information needs to persist?
4.  **Human-in-the-Loop (HITL):** Describe how human intervention will be integrated into your design. At what stage(s) would a human be involved, and what would their role be?

#### Part 2: Implementation (Simplified)

1.  **Agent Implementation:** Using Python, implement the core logic for at least **two** of your designed agents. You don't need to implement all tools fully; mock functions are sufficient.
2.  **Orchestrator Implementation:** Create a simplified orchestrator function or class that manages the flow between your implemented agents for a given ticket.
3.  **Simulation:** Utilize the provided `MockLLM`, `MockTool`, and `Ticket` dataclass to simulate the environment and agent interactions.
4.  **Demonstration:** Run your workflow with the provided `mock_tickets` to demonstrate its functionality and how the ticket state evolves.

### Requirements:

*   Your design must clearly define agent roles, responsibilities, and their interactions.
*   Your implementation should demonstrate the flow for at least one full cycle (or a partial cycle if human-in-the-loop is involved).
*   State updates must be explicit and passed between agents.
*   Include comments in your code explaining your design choices and any production considerations.

### Evaluation Criteria:

*   **Design Clarity & Completeness (40%):** How well are agent roles, responsibilities, and interactions defined? Is the workflow logical and comprehensive?
*   **Production Readiness Considerations (30%):** How effectively are state management, human-in-the-loop, and potential error handling discussed/integrated?
*   **Implementation Correctness & Readability (30%):** Does the code correctly reflect the design? Is it well-structured, commented, and easy to understand?


In [ ]:
import dataclasses
import enum
import time
import random
from typing import List, Dict, Any, Optional, Callable

# --- Mock Components for Simulation ---

class MockLLM:
    """A mock Large Language Model to simulate agent reasoning and content generation."""
    def __init__(self, model_name: str = "gpt-4o-2026-preview"):
        self.model_name = model_name

    def chat_completion(self, prompt: str, temperature: float = 0.7) -> str:
        """Simulates an LLM chat completion based on the prompt."""
        print(f"  [MockLLM] Processing prompt (model: {self.model_name})...")
        time.sleep(0.5) # Simulate API call latency
        if "categorize" in prompt.lower():
            categories = ["Billing", "Technical Bug", "Feature Request", "General Inquiry", "Account Management"]
            return random.choice(categories)
        elif "priority" in prompt.lower():
            priorities = ["Low", "Medium", "High", "Critical"]
            return random.choice(priorities)
        elif "search query" in prompt.lower():
            return f"keywords for search: {prompt.split('issue:')[-1].strip().split('.')[0]}"
        elif "draft resolution" in prompt.lower():
            return f"Drafted resolution for: {prompt.split('issue:')[-1].strip().split('.')[0]}. Based on gathered info: {prompt.split('info:')[-1].strip().split('---')[0]}"
        elif "summary" in prompt.lower():
            return f"Summary of issue: {prompt[:50]}..."
        return f"Mock LLM response for: {prompt[:100]}..."

class MockTool:
    """A mock external tool to simulate database lookups, API calls, etc."""
    def __init__(self, name: str):
        self.name = name

    def knowledge_base_search(self, query: str) -> str:
        """Simulates searching a knowledge base for relevant articles."""
        print(f"  [MockTool: {self.name}] Searching knowledge base for '{query}'...")
        time.sleep(0.3)
        if "billing" in query.lower():
            return "Found article: 'Understanding Your Invoice' and 'Payment Methods'."
        elif "login" in query.lower() or "password" in query.lower():
            return "Found article: 'Troubleshooting Login Issues' and 'Resetting Your Password'."
        elif "bug" in query.lower():
            return "Found article: 'Reporting a Technical Bug' and 'Known Issues List'."
        return "No specific articles found, general troubleshooting tips available."

    def crm_lookup(self, customer_id: str) -> Dict[str, Any]:
        """Simulates looking up customer information in a CRM system."""
        print(f"  [MockTool: {self.name}] Looking up CRM for customer ID '{customer_id}'...")
        time.sleep(0.2)
        mock_data = {
            "CUST001": {"name": "Alice Smith", "tier": "Premium", "recent_orders": 3},
            "CUST002": {"name": "Bob Johnson", "tier": "Standard", "recent_orders": 1},
            "CUST003": {"name": "Charlie Brown", "tier": "Basic", "recent_orders": 0}
        }
        return mock_data.get(customer_id, {"name": "Unknown", "tier": "N/A", "recent_orders": 0})

# --- Ticket State Management ---

class TicketStatus(enum.Enum):
    NEW = "New"
    TRIAGED = "Triaged"
    INFO_GATHERED = "Information Gathered"
    SOLUTION_DRAFTED = "Solution Drafted"
    PENDING_HUMAN_REVIEW = "Pending Human Review"
    RESOLVED = "Resolved"
    ESCALATED = "Escalated"
    CLOSED = "Closed"

class TicketPriority(enum.Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"

@dataclasses.dataclass
class Ticket:
    id: str
    customer_id: str
    subject: str
    description: str
    status: TicketStatus = TicketStatus.NEW
    category: Optional[str] = None
    priority: Optional[TicketPriority] = None
    resolution_draft: Optional[str] = None
    gathered_info: List[str] = dataclasses.field(default_factory=list)
    history: List[str] = dataclasses.field(default_factory=list)
    assigned_to_human: bool = False
    human_review_notes: Optional[str] = None

    def add_history(self, entry: str):
        self.history.append(f"[{time.strftime('%H:%M:%S')}] {entry}")

    def __str__(self):
        return (f"Ticket ID: {self.id}\n"\
                f"  Customer: {self.customer_id}\n"\
                f"  Subject: {self.subject}\n"\
                f"  Status: {self.status.value}\n"\
                f"  Category: {self.category or 'N/A'}\n"\
                f"  Priority: {self.priority.value if self.priority else 'N/A'}\n"\
                f"  Resolution Draft: {self.resolution_draft or 'N/A'}\n"\
                f"  Assigned to Human: {self.assigned_to_human}\n"\
                f"  History: {len(self.history)} entries")

# --- Sample Tickets ---
mock_tickets: List[Ticket] = [
    Ticket(
        id="TKT001",
        customer_id="CUST001",
        subject="Cannot log in to my account",
        description="I'm unable to log in. I've tried resetting my password multiple times, but it keeps saying invalid credentials. My email is alice@example.com."
    ),
    Ticket(
        id="TKT002",
        customer_id="CUST002",
        subject="Billing inquiry - incorrect charge",
        description="My last bill shows a charge for a premium feature I never subscribed to. Please investigate and correct this."
    ),
    Ticket(
        id="TKT003",
        customer_id="CUST003",
        subject="Feature Request: Dark Mode",
        description="I would love to see a dark mode option for your application. It would greatly improve user experience, especially at night."
    ),
    Ticket(
        id="TKT004",
        customer_id="CUST001",
        subject="Critical Bug: Data Loss on Save",
        description="When I save my project, sometimes all my data disappears. This is a critical issue and I've lost hours of work!"
    )
]

print("Setup complete. Mock LLM, Tools, and Ticket dataclass are ready.")


## Your Implementation

Now it's your turn! Based on the exercise description above, design and implement your multi-agent workflow for customer support ticket resolution.

1.  **Define your agent classes:** Create Python classes for each agent you identified in your design phase (e.g., `TriageAgent`, `InformationGatheringAgent`, `SolutionDraftingAgent`). Each agent should have a method (e.g., `process_ticket`) that takes a `Ticket` object as input, performs its designated task, and updates the ticket's state.
2.  **Implement agent logic:** Inside each agent's `process_ticket` method, use the provided `MockLLM` and `MockTool` instances to simulate agent reasoning and external tool interactions. Remember to update the `ticket.status`, `ticket.category`, `ticket.priority`, `ticket.resolution_draft`, and `ticket.history` as appropriate.
3.  **Create an orchestrator:** Develop an orchestrator function or class that instantiates your agents and manages the sequential flow of a ticket through the different stages of your workflow. Include conditional logic for human review or escalation.
4.  **Demonstrate:** Run your orchestrator with the `mock_tickets` provided in the setup cell. Print the state of each ticket before and after the workflow to show the changes.

Feel free to extend the `Ticket` dataclass or add more mock tools as needed to support your design. Focus on clarity, modularity, and demonstrating the agentic patterns.


In [ ]:
import dataclasses
import enum
import time
import random
from typing import List, Dict, Any, Optional, Callable

# Re-importing for self-contained solution, assuming the setup cell might not always be run first
# In a real notebook, these would be defined once.

class MockLLM:
    """A mock Large Language Model to simulate agent reasoning and content generation."""
    def __init__(self, model_name: str = "gpt-4o-2026-preview"):
        self.model_name = model_name

    def chat_completion(self, prompt: str, temperature: float = 0.7) -> str:
        """Simulates an LLM chat completion based on the prompt."""
        # print(f"  [MockLLM] Processing prompt (model: {self.model_name})...")
        time.sleep(0.05) # Simulate API call latency
        if "categorize" in prompt.lower():
            categories = ["Billing", "Technical Bug", "Feature Request", "General Inquiry", "Account Management"]
            return random.choice(categories)
        elif "priority" in prompt.lower():
            priorities = ["Low", "Medium", "High", "Critical"]
            return random.choice(priorities)
        elif "search query" in prompt.lower():
            return f"keywords for search: {prompt.split('issue:')[-1].strip().split('.')[0]}"
        elif "draft resolution" in prompt.lower():
            return f"Drafted resolution for: {prompt.split('issue:')[-1].strip().split('.')[0]}. Based on gathered info: {prompt.split('info:')[-1].strip().split('---')[0]}"
        elif "summary" in prompt.lower():
            return f"Summary of issue: {prompt[:50]}..."
        return f"Mock LLM response for: {prompt[:100]}..."

class MockTool:
    """A mock external tool to simulate database lookups, API calls, etc."""
    def __init__(self, name: str):
        self.name = name

    def knowledge_base_search(self, query: str) -> str:
        """Simulates searching a knowledge base for relevant articles."""
        # print(f"  [MockTool: {self.name}] Searching knowledge base for '{query}'...")
        time.sleep(0.03)
        if "billing" in query.lower():
            return "Found article: 'Understanding Your Invoice' and 'Payment Methods'."
        elif "login" in query.lower() or "password" in query.lower():
            return "Found article: 'Troubleshooting Login Issues' and 'Resetting Your Password'."
        elif "bug" in query.lower():
            return "Found article: 'Reporting a Technical Bug' and 'Known Issues List'."
        return "No specific articles found, general troubleshooting tips available."

    def crm_lookup(self, customer_id: str) -> Dict[str, Any]:
        """Simulates looking up customer information in a CRM system."""
        # print(f"  [MockTool: {self.name}] Looking up CRM for customer ID '{customer_id}'...")
        time.sleep(0.02)
        mock_data = {
            "CUST001": {"name": "Alice Smith", "tier": "Premium", "recent_orders": 3},
            "CUST002": {"name": "Bob Johnson", "tier": "Standard", "recent_orders": 1},
            "CUST003": {"name": "Charlie Brown", "tier": "Basic", "recent_orders": 0}
        }
        return mock_data.get(customer_id, {"name": "Unknown", "tier": "N/A", "recent_orders": 0})

class TicketStatus(enum.Enum):
    NEW = "New"
    TRIAGED = "Triaged"
    INFO_GATHERED = "Information Gathered"
    SOLUTION_DRAFTED = "Solution Drafted"
    PENDING_HUMAN_REVIEW = "Pending Human Review"
    RESOLVED = "Resolved"
    ESCALATED = "Escalated"
    CLOSED = "Closed"

class TicketPriority(enum.Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"

@dataclasses.dataclass
class Ticket:
    id: str
    customer_id: str
    subject: str
    description: str
    status: TicketStatus = TicketStatus.NEW
    category: Optional[str] = None
    priority: Optional[TicketPriority] = None
    resolution_draft: Optional[str] = None
    gathered_info: List[str] = dataclasses.field(default_factory=list)
    history: List[str] = dataclasses.field(default_factory=list)
    assigned_to_human: bool = False
    human_review_notes: Optional[str] = None

    def add_history(self, entry: str):
        self.history.append(f"[{time.strftime('%H:%M:%S')}] {entry}")

    def __str__(self):
        return (f"Ticket ID: {self.id}\n"\
                f"  Customer: {self.customer_id}\n"\
                f"  Subject: {self.subject}\n"\
                f"  Status: {self.status.value}\n"\
                f"  Category: {self.category or 'N/A'}\n"\
                f"  Priority: {self.priority.value if self.priority else 'N/A'}\n"\
                f"  Resolution Draft: {self.resolution_draft or 'N/A'}\n"\
                f"  Assigned to Human: {self.assigned_to_human}\n"\
                f"  History: {len(self.history)} entries")

# --- Sample Tickets (re-defined for self-contained solution) ---
mock_tickets: List[Ticket] = [
    Ticket(
        id="TKT001",
        customer_id="CUST001",
        subject="Cannot log in to my account",
        description="I'm unable to log in. I've tried resetting my password multiple times, but it keeps saying invalid credentials. My email is alice@example.com."
    ),
    Ticket(
        id="TKT002",
        customer_id="CUST002",
        subject="Billing inquiry - incorrect charge",
        description="My last bill shows a charge for a premium feature I never subscribed to. Please investigate and correct this."
    ),
    Ticket(
        id="TKT003",
        customer_id="CUST003",
        subject="Feature Request: Dark Mode",
        description="I would love to see a dark mode option for your application. It would greatly improve user experience, especially at night."
    ),
    Ticket(
        id="TKT004",
        customer_id="CUST001",
        subject="Critical Bug: Data Loss on Save",
        description="When I save my project, sometimes all my data disappears. This is a critical issue and I've lost hours of work!"
    )
]


# --- Reference Solution --- 

# Part 1: Design (Conceptual - as comments within the code structure)

# Agent Roles:
# 1. TriageAgent: Categorizes and prioritizes incoming tickets.
#    - Capabilities: LLM for natural language understanding, classification.
#    - Input: Raw Ticket object (subject, description).
#    - Output: Updated Ticket object (category, priority, status).
#
# 2. InformationGatheringAgent: Gathers relevant context for the ticket.
#    - Capabilities: LLM for query generation, Knowledge Base Search tool, CRM Lookup tool.
#    - Input: Triaged Ticket object.
#    - Output: Updated Ticket object (gathered_info, status).
#
# 3. SolutionDraftingAgent: Formulates a preliminary resolution or response.
#    - Capabilities: LLM for content generation.
#    - Input: Ticket object with gathered_info.
#    - Output: Updated Ticket object (resolution_draft, status).
#
# 4. HumanReviewAgent (Conceptual): Represents the human-in-the-loop for critical decisions.
#    - Capabilities: Human judgment, ability to approve/reject/escalate.
#    - Input: Ticket object with resolution_draft.
#    - Output: Updated Ticket object (status, assigned_to_human, human_review_notes).

# Workflow Orchestration:
# New Ticket -> TriageAgent -> InformationGatheringAgent -> SolutionDraftingAgent -> 
# (Conditional) -> HumanReviewAgent (if High/Critical priority or specific category) -> 
# (If approved by Human) -> Resolve Ticket
# (If rejected/escalated by Human) -> Escalate Ticket
# (If no Human Review needed) -> Resolve Ticket

# State Management:
# The `Ticket` dataclass serves as the central state object. Each agent receives the current state of the ticket,
# performs its operation, and returns an updated `Ticket` object. This ensures all relevant information
# (category, priority, resolution_draft, history, etc.) is consistently available and mutable across the workflow.
# In a production system, this state would be persisted in a database (e.g., PostgreSQL, MongoDB) or a state store
# (e.g., Redis) and passed via message queues or API calls between microservices representing agents.

# Human-in-the-Loop (HITL):
# Integrated after solution drafting. For tickets flagged as 'High' or 'Critical' priority, or specific categories
# like 'Billing' or 'Critical Bug', the workflow pauses and assigns the ticket for human review. The human can then
# approve the draft, modify it, or escalate the ticket further. This is simulated by a simple `input()` prompt.


# Part 2: Implementation

class BaseAgent:
    """Abstract base class for agents."""
    def __init__(self, name: str, llm: MockLLM, tools: Dict[str, MockTool] = None):
        self.name = name
        self.llm = llm
        self.tools = tools if tools is not None else {}
        print(f"Initialized {self.name}.")

    def process_ticket(self, ticket: Ticket) -> Ticket:
        raise NotImplementedError


class TriageAgent(BaseAgent):
    """Agent responsible for categorizing and prioritizing incoming tickets."""
    def __init__(self, llm: MockLLM):
        super().__init__("TriageAgent", llm)

    def process_ticket(self, ticket: Ticket) -> Ticket:
        print(f"[{self.name}] Processing Ticket {ticket.id}: {ticket.subject}")
        
        # Use LLM to determine category
        category_prompt = f"Categorize the following customer support issue: Subject: {ticket.subject}. Description: {ticket.description}. Choose from: Billing, Technical Bug, Feature Request, General Inquiry, Account Management."
        category_response = self.llm.chat_completion(category_prompt)
        ticket.category = category_response.strip()
        
        # Use LLM to determine priority
        priority_prompt = f"Assign a priority (Low, Medium, High, Critical) to this issue based on its description and category '{ticket.category}': Subject: {ticket.subject}. Description: {ticket.description}."
        priority_response = self.llm.chat_completion(priority_prompt)
        try:
            ticket.priority = TicketPriority[priority_response.strip().upper()]
        except KeyError:
            ticket.priority = TicketPriority.MEDIUM # Default if LLM gives unexpected output

        ticket.status = TicketStatus.TRIAGED
        ticket.add_history(f"Categorized as '{ticket.category}' with priority '{ticket.priority.value}'.")
        print(f"[{self.name}] Ticket {ticket.id} triaged: Category={ticket.category}, Priority={ticket.priority.value}")
        return ticket


class InformationGatheringAgent(BaseAgent):
    """Agent responsible for gathering relevant information using tools."""
    def __init__(self, llm: MockLLM, knowledge_base_tool: MockTool, crm_tool: MockTool):
        super().__init__("InformationGatheringAgent", llm, {"kb": knowledge_base_tool, "crm": crm_tool})

    def process_ticket(self, ticket: Ticket) -> Ticket:
        print(f"[{self.name}] Processing Ticket {ticket.id}: Gathering information.")
        
        gathered_info_list = []

        # CRM Lookup
        customer_info = self.tools["crm"].crm_lookup(ticket.customer_id)
        gathered_info_list.append(f"CRM Info: {customer_info}")
        ticket.add_history(f"Retrieved CRM info for customer {ticket.customer_id}.")

        # Knowledge Base Search
        search_query_prompt = f"Generate a concise search query for a knowledge base based on the following issue: Category: {ticket.category}, Subject: {ticket.subject}, Description: {ticket.description}."
        search_query = self.llm.chat_completion(search_query_prompt)
        kb_results = self.tools["kb"].knowledge_base_search(search_query)
        gathered_info_list.append(f"Knowledge Base Results: {kb_results}")
        ticket.add_history(f"Searched knowledge base with query '{search_query}'.")

        ticket.gathered_info = gathered_info_list
        ticket.status = TicketStatus.INFO_GATHERED
        print(f"[{self.name}] Ticket {ticket.id} information gathered.")
        return ticket


class SolutionDraftingAgent(BaseAgent):
    """Agent responsible for drafting a resolution based on gathered information."""
    def __init__(self, llm: MockLLM):
        super().__init__("SolutionDraftingAgent", llm)

    def process_ticket(self, ticket: Ticket) -> Ticket:
        print(f"[{self.name}] Processing Ticket {ticket.id}: Drafting solution.")
        
        draft_prompt = f"Draft a concise customer-facing resolution for the following issue. \n\nIssue: {ticket.subject} - {ticket.description}\nCategory: {ticket.category}\nPriority: {ticket.priority.value}\nGathered Info: {'; '.join(ticket.gathered_info)}\n\nResolution Draft:"
        
        resolution_draft = self.llm.chat_completion(draft_prompt)
        ticket.resolution_draft = resolution_draft.strip()
        ticket.status = TicketStatus.SOLUTION_DRAFTED
        ticket.add_history(f"Drafted initial resolution: {ticket.resolution_draft[:100]}...")
        print(f"[{self.name}] Ticket {ticket.id} solution drafted.")
        return ticket


class AgenticWorkflowOrchestrator:
    """Orchestrates the multi-agent workflow for customer support tickets."""
    def __init__(self, llm: MockLLM):
        self.llm = llm
        self.triage_agent = TriageAgent(llm=self.llm)
        self.info_gathering_agent = InformationGatheringAgent(
            llm=self.llm,
            knowledge_base_tool=MockTool("KnowledgeBase"),
            crm_tool=MockTool("CRM")
        )
        self.solution_drafting_agent = SolutionDraftingAgent(llm=self.llm)
        print("Orchestrator initialized with agents.")

    def run_workflow(self, ticket: Ticket) -> Ticket:
        print(f"\n--- Starting Workflow for Ticket {ticket.id} ---")
        ticket.add_history("Workflow started.")

        # 1. Triage
        ticket = self.triage_agent.process_ticket(ticket)
        
        # 2. Information Gathering
        ticket = self.info_gathering_agent.process_ticket(ticket)

        # 3. Solution Drafting
        ticket = self.solution_drafting_agent.process_ticket(ticket)

        # 4. Human Review / Escalation Logic (Human-in-the-Loop)
        if ticket.priority in [TicketPriority.HIGH, TicketPriority.CRITICAL] or ticket.category in ["Billing", "Technical Bug"]:
            ticket.status = TicketStatus.PENDING_HUMAN_REVIEW
            ticket.assigned_to_human = True
            ticket.add_history("Flagged for human review due to priority/category.")
            print(f"[Orchestrator] Ticket {ticket.id} flagged for human review. Priority: {ticket.priority.value}, Category: {ticket.category}")
            
            # Simulate human interaction
            print(f"  Human Review Needed for Ticket {ticket.id}:\n  Subject: {ticket.subject}\n  Draft: {ticket.resolution_draft}")
            human_decision = input("  Human: Approve (A), Reject (R), or Escalate (E)? ").strip().upper()
            
            if human_decision == 'A':
                ticket.status = TicketStatus.RESOLVED
                ticket.assigned_to_human = False
                ticket.human_review_notes = "Approved by human."
                ticket.add_history("Human approved the drafted resolution.")
                print(f"[Orchestrator] Ticket {ticket.id} approved and resolved by human.")
            elif human_decision == 'R':
                ticket.status = TicketStatus.SOLUTION_DRAFTED # Back to drafting or re-triage
                ticket.assigned_to_human = False
                ticket.human_review_notes = "Rejected by human, needs re-drafting."
                ticket.add_history("Human rejected the drafted resolution. Needs re-drafting.")
                print(f"[Orchestrator] Ticket {ticket.id} rejected by human. Re-routing for further work.")
                # For simplicity, we'll just mark it as rejected here. In a real system, it would loop back.
            elif human_decision == 'E':
                ticket.status = TicketStatus.ESCALATED
                ticket.assigned_to_human = True
                ticket.human_review_notes = "Escalated by human to a specialist."
                ticket.add_history("Human escalated the ticket to a specialist.")
                print(f"[Orchestrator] Ticket {ticket.id} escalated by human.")
            else:
                print("  Invalid human input. Defaulting to pending review.")
                ticket.add_history("Invalid human input, remaining pending human review.")

        else:
            # Auto-resolve for lower priority/routine tickets
            ticket.status = TicketStatus.RESOLVED
            ticket.add_history("Auto-resolved based on low priority/routine category.")
            print(f"[Orchestrator] Ticket {ticket.id} auto-resolved.")

        ticket.add_history("Workflow completed.")
        print(f"--- Workflow Finished for Ticket {ticket.id} ---")
        return ticket


# --- Demonstration ---

# Initialize LLM and Orchestrator
llm_instance = MockLLM()
orchestrator = AgenticWorkflowOrchestrator(llm=llm_instance)

processed_tickets = []
for i, ticket in enumerate(mock_tickets):
    print(f"\n### Processing Ticket {ticket.id} ({i+1}/{len(mock_tickets)}) ###")
    print("Initial State:")
    print(ticket)
    
    final_ticket_state = orchestrator.run_workflow(ticket)
    processed_tickets.append(final_ticket_state)
    
    print("\nFinal State:")
    print(final_ticket_state)
    print("--------------------------------------------------")

print("\nAll tickets processed.")

# --- Production Considerations (beyond this exercise) ---
# 1.  **Frameworks:** For real-world applications, consider robust agentic frameworks like LangGraph, CrewAI, AutoGen, or Marvin. These provide abstractions for agent definition, tool integration, state management, and orchestration.
# 2.  **State Persistence:** The `Ticket` object's state should be persisted in a database (e.g., PostgreSQL, MongoDB) after each agent's step to ensure durability and recoverability, especially for long-running workflows.
# 3.  **Asynchronous Processing & Message Queues:** Agents should ideally run asynchronously. Use message queues (e.g., Kafka, RabbitMQ, AWS SQS) to pass tickets/messages between agents, decoupling them and enabling scalable, fault-tolerant processing.
# 4.  **Error Handling & Retries:** Implement robust error handling, including retries for transient failures, dead-letter queues for unprocessable messages, and fallback mechanisms (e.g., immediate human escalation if an agent fails repeatedly).
# 5.  **Observability:** Integrate comprehensive logging, tracing (e.g., OpenTelemetry), and monitoring (e.g., Prometheus, Grafana) to understand agent behavior, identify bottlenecks, and debug issues in production.
# 6.  **Security:** Secure API keys for LLMs and external tools. Implement input validation and sanitization to prevent prompt injection or other security vulnerabilities.
# 7.  **Tool Orchestration:** For complex tool use, consider a dedicated Tool Agent or a tool router that intelligently selects and invokes the right tools based on agent requests.
# 8.  **Human-in-the-Loop UI:** For HITL, integrate with a dedicated UI or task management system where human agents can review, approve, or modify agent outputs.
